# US 11 — Baseline tree model (Random Forest)

**Sprint 5** | Same train/test as US 9 and US 10 (`docs/modeling_spec.md`)

Run **Run All** from repo root or `notebooks/modeling/`.

**Output:** `reports/us11_tree_metrics.csv`, `reports/figures/us11_feature_importance.png`

In [ ]:
from pathlib import Path

import pandas as pd


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "processed").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data" / "processed").is_dir():
        return cwd.parent
    if cwd.name == "modeling" and (cwd.parent.parent / "data" / "processed").is_dir():
        return cwd.parent.parent
    return cwd


REPO_ROOT = _repo_root()
TRAIN_PATH = REPO_ROOT / "data" / "processed" / "train.csv"
TEST_PATH = REPO_ROOT / "data" / "processed" / "test.csv"
METRICS_PATH = REPO_ROOT / "reports" / "us11_tree_metrics.csv"
FIG_PATH = REPO_ROOT / "reports" / "figures" / "us11_feature_importance.png"

TARGET = "electricity_demand_per_capita"
FEATURES = [
    "temperature_change_c",
    "co2_per_capita",
    "gdp",
    "population",
    "renewables_share_elec",
    "fossil_share_elec",
]

print("Repo root:", REPO_ROOT)
print("Train:", TRAIN_PATH)
print("Test:", TEST_PATH)

## Task 1 — Load `train.csv` and `test.csv`

In [ ]:
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "Missing train.csv or test.csv. Pull latest main (US 9) before running US 11."
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Task 1 DONE")
print("Train:", train.shape)
print("Test:", test.shape)
train.head(3)

## Task 2 — Separate X and y

In [ ]:
X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]
print("Task 2 DONE")

## Task 3 — Train Random Forest (train set only)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Task 3 DONE — Random Forest trained")

## Task 4 & 5 — Predict on test and calculate metrics

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Task 4 & 5 DONE — test metrics")
print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R²:", round(r2, 4))

## Task 6 — Feature importance plot

In [ ]:
import matplotlib.pyplot as plt

FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Random Forest — feature importance (US 11)")
ax.set_xlabel("Importance")
plt.tight_layout()
fig.savefig(FIG_PATH, dpi=120)
plt.show()
print("Task 6 DONE — saved:", FIG_PATH)

## Task 7 — Save metrics CSV

In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{
    "model": "random_forest",
    "RMSE": rmse,
    "MAE": mae,
    "R2": r2,
}]).to_csv(METRICS_PATH, index=False)

print("Task 7 DONE — saved:", METRICS_PATH)
print("Next step (US 12): compare with reports/us10_linear_metrics.csv")